# GloVe from Scratch — WikiText-2

End-to-end interface to the implementation in `src/`. The algorithm itself
(sparse co-occurrence, weighted least-squares objective, analytic gradients,
AdaGrad) lives in the source modules; this notebook only orchestrates and
inspects.

**Colab:** `Runtime → Change runtime type → Hardware accelerator: GPU`
(whatever GPU is offered — T4, L4, etc.). The code falls back to CPU
automatically and never needs more than ~4 GB of VRAM.

## 0. Setup

In [ ]:
# On Colab, clone/upload the project first, then:
# !pip install -q -r requirements.txt

import os, sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists(os.path.join(PROJECT_ROOT, 'src')):
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('project root:', PROJECT_ROOT)

In [ ]:
import config as project_config
from src.utils import set_seed, resolve_device, print_device_banner

project_config.ensure_dirs()
CONFIG = project_config.get_config()
set_seed(CONFIG['seed'])
device = resolve_device(CONFIG['device'])
print_device_banner(device)
CONFIG

## 1. Toy corpus validation (run this before the real corpus)

Verifies the window behaviour, symmetry, distance weighting and accumulation
on `"the cat sat on the mat"`, where every entry can be checked by hand.

In [ ]:
from src.preprocessing import tokenize, build_vocabulary
from src.cooccurrence import build_from_text

toy_tokens = tokenize('the cat sat on the mat')
toy_vocab = build_vocabulary(toy_tokens, min_frequency=1, max_vocab_size=50, verbose=False)
print(toy_tokens)

for window in (1, 2):
    toy_matrix = build_from_text(toy_tokens, toy_vocab.word_to_idx, window_size=window)
    print(f'\n--- window_size = {window} ---')
    for (i, j), value in sorted(toy_matrix.to_dict().items()):
        wi, wj = toy_vocab.idx_to_word[i], toy_vocab.idx_to_word[j]
        print(f'X[{wi:>4}, {wj:<4}] = {value:.3f}')

In [ ]:
# Full unit-test suite (tokenizer, vocab, co-occurrence, f(x), loss, gradients,
# AdaGrad, analogy solver). Gradients are checked against autograd.
!python tests/test_glove.py

## 2. Corpus, vocabulary and sparse co-occurrence

In [ ]:
from src.data import load_wikitext2_text
from src.experiments import CorpusCache

corpus = CorpusCache(load_wikitext2_text('train'))
print(f'tokens: {len(corpus):,}')
print(corpus.tokens[:25])

## 3. Tiny synthetic training sanity check

A few epochs on `"king man queen woman"` repeated: the loss must stay finite,
decrease, and the parameters must actually move before we touch the big corpus.

In [ ]:
import numpy as np, torch
from src.glove import GloVeModel

tiny_tokens = tokenize('king man queen woman ' * 4)
tiny_vocab = build_vocabulary(tiny_tokens, 1, 10, verbose=False)
tiny_matrix = build_from_text(tiny_tokens, tiny_vocab.word_to_idx, window_size=2)
tiny_model = GloVeModel(len(tiny_vocab), embedding_dim=8, seed=42)

ti = torch.from_numpy(tiny_matrix.rows.astype(np.int64))
tj = torch.from_numpy(tiny_matrix.cols.astype(np.int64))
tx = torch.from_numpy(tiny_matrix.values.astype(np.float32))
for epoch in range(10):
    loss = tiny_model.train_batch(ti, tj, tx, CONFIG['learning_rate']) / len(tx)
    print(f'epoch {epoch:02d}  loss {loss:.6f}')
print('all finite:', bool(torch.isfinite(tiny_model.embeddings()).all()))

## 4. Smoke test on WikiText-2 (2 epochs)

Confirms the GPU path, memory safety and a decreasing loss before committing
to the full run.

In [ ]:
from src.experiments import run_pipeline

smoke_cfg = project_config.get_config(epochs=2, checkpoint_every=0)
smoke = run_pipeline(corpus.tokens, smoke_cfg, device, name='smoke')
print('final loss:', smoke.training['final_loss'])
print('peak GPU MB:', smoke.training['peak_gpu_memory_mb'])
del smoke

## 5. Baseline training (100D, window 1, min_freq 5, 30 epochs)

Checkpoints are written every 5 epochs to `checkpoints/`; pass `resume=True`
to continue an interrupted run from the latest one.

In [ ]:
result = run_pipeline(
    corpus.tokens, CONFIG, device,
    name='custom-wikitext2',
    checkpoint_dir=project_config.PATHS['checkpoints'],
    resume=False,
)
print('final loss :', result.training['final_loss'])
print('time (s)   :', round(result.training['training_time_s'], 2))
print('peak GPU MB:', result.training['peak_gpu_memory_mb'])

In [ ]:
import os
from src.visualization import plot_loss_curve
from IPython.display import Image as ShowImage

history = result.training['history'].to_dict()
path = plot_loss_curve(history['epoch'], history['loss'],
                       os.path.join(project_config.PATHS['results'], 'loss_curve.png'))
ShowImage(filename=path)

## 6. Final embeddings (W + W_context) and export

In [ ]:
from src.experiments import export_vectors

export_vectors(result, project_config.PATHS['vectors'])

## 7. Nearest neighbours and analogies

In [ ]:
from src.evaluation import select_query_words, evaluate_nearest_neighbors, evaluate_analogies, oov_analysis

queries, missing = select_query_words(result.index,
                                      project_config.NEAREST_NEIGHBOR_QUERIES,
                                      project_config.NEIGHBOR_FALLBACKS, count=5)
print('queries:', queries, '| replaced (OOV):', missing)
for query in queries:
    top = result.index.nearest_neighbors(query, 10)
    print(f"{query:>12}: " + ', '.join(f'{w} ({s:.2f})' for w, s in top))

In [ ]:
import pandas as pd

rows, analogy_summary = evaluate_analogies(result.index, project_config.ANALOGIES)
coverage = oov_analysis(result.index, project_config.ANALOGIES, queries)
display(pd.DataFrame(rows)[['a', 'b', 'c', 'expected', 'predicted', 'correct', 'similarity']])
print(analogy_summary)
print('OOV rate %:', round(coverage['oov_rate_percent'], 2))

## 8. PCA visualisation

In [ ]:
from src.visualization import plot_pca_embeddings, select_visualization_words

words = select_visualization_words(
    result.index,
    list(project_config.NEAREST_NEIGHBOR_QUERIES) + list(project_config.NEIGHBOR_FALLBACKS)
    + [w for quad in project_config.ANALOGIES for w in quad], count=80)
pca_path = plot_pca_embeddings(result.index, words,
                               os.path.join(project_config.PATHS['results'], 'pca_embeddings.png'))
ShowImage(filename=pca_path)

## 9. Comparison with official glove.6B.100d

Downloaded **after** training and used for evaluation only — never to
initialise or fine-tune our model.

In [ ]:
from src.pretrained import load_pretrained_index

official = load_pretrained_index()
official_rows, official_summary = evaluate_analogies(official, project_config.ANALOGIES)
official_cov = oov_analysis(official, project_config.ANALOGIES, queries)

for query in queries:
    mine = [w for w, _ in result.index.nearest_neighbors(query, 5)]
    theirs = [w for w, _ in official.nearest_neighbors(query, 5)]
    print(f'{query:>12} | ours: {mine}\n{"":>12} | 6B  : {theirs}')

print('\ncustom accuracy  :', round(analogy_summary["accuracy"] * 100, 2), '%')
print('official accuracy:', round(official_summary["accuracy"] * 100, 2), '%')
print('custom OOV %     :', round(coverage["oov_rate_percent"], 2))
print('official OOV %   :', round(official_cov["oov_rate_percent"], 2))

## 10. Ablations and CPU/GPU benchmark (optional, expensive)

In [ ]:
from src.experiments import run_ablations, run_cpu_gpu_benchmark

RUN_ABLATIONS = project_config.RUN_ABLATIONS
RUN_CPU_BENCHMARK = project_config.RUN_CPU_BENCHMARK

if RUN_ABLATIONS:
    ablation_rows = run_ablations(corpus.tokens, CONFIG, device)
    display(pd.DataFrame(ablation_rows))

if RUN_CPU_BENCHMARK:
    display(pd.DataFrame(run_cpu_gpu_benchmark(corpus.tokens, CONFIG, epochs=3)))

## 11. One-command full pipeline

Everything above, plus all CSVs, plots and the PDF report:

In [ ]:
# !python run.py --cpu-benchmark
pd.read_csv(os.path.join(project_config.PATHS['results'], 'final_summary.csv')).T